# Data reliability lab: how could input errors change my conclusions?

**Research question:** how do time precision, quantity, instrument choice and
unreliable files affect an otherwise simple analysis? These are small **synthetic**
examples. They test reasoning and software behavior, not an empirical strategy.
No provider is contacted; the notebook uses temporary files and child processes.

| Concern | Read these sections | What to look for |
| --- | --- | --- |
| A decision accidentally uses unavailable information | 2, 4–5 | Compare source/capture clocks and decision cutoffs. |
| An amount has the wrong units or rounding | 4, 6–7 | Compare exact notionals, rejected instruments and the hand ledger. |
| An experiment cannot be reproduced | 3, 8 | Retain implementation/input identity and findings. |
| A download is missing, altered or interrupted | 10–15 | Distinguish rejected input from completed and recoverable work. |

Read the questions and resulting tables first. Sources of implementation-heavy
cells are collapsed where supported. **Execute all cells in order from a fresh
kernel**, even if you are reading one topic: the examples share setup. The named
scenario cell remains visible so its adjustable assumptions are easy to find.

Use [historical trade-data exploration](research-foundations.ipynb) for actual
Binance rows, including the MR 08 empty/missing archive examples. This lab retains
numbered foundation and MR 00–07 checks as supporting evidence. Fixed reference
calculations stay separate from editable scenarios; they do not tune the frozen
[P1 hypothesis](../docs/research/p1-baseline.md).

[Researcher's guide](README.md) · [Execution and troubleshooting](notebook-operations.md).
Explanations were reorganized on 2026-09-16; executable cells, their order and
previous outputs retain the original execution evidence.

Source cells use the MR 28 functional APIs. Retained outputs describe their recorded revisions and have not been regenerated for this layout refactor.


## 1. Prepare a reproducible offline study


Use the pinned interpreter and installed package. Presentation helpers stay local;
domain identities, clocks, decoding and financial operations come from the package.
Interactive use: select the repository's `.venv/bin/python` kernel, then **Restart
Kernel → Run All**. The notebook uses the installed package; it does not patch
`sys.path`. `nbclient` and `nbformat` belong to the separate execution runner and
need not be installed in the interactive kernel. The environment table records
only notebook tools actually present in that kernel.

In [1]:
# standard library imports
import hashlib
import json
import os
import platform
import subprocess
import sys
from datetime import UTC, datetime, timedelta
from decimal import ROUND_HALF_UP, Decimal
from html import escape
from importlib.metadata import distributions
from pathlib import Path
from uuid import UUID

# third party library imports
from IPython.display import HTML, Markdown, display

# project library imports
import market_data
from financial_units import Currency, Dimension, Money, Price, Quantity, QuantityKind, UnitError
from market_data.adapters.providers.tardis import TardisDecoder
from market_data.adapters.sources import SourceAdapters
from market_data.adapters.venues.binance import BinancePayloadDecoder
from market_data.contracts import Disconnect, MarketEventError, Trade
from market_data.datasets import DatasetPublication
from market_data.storage import RawFiles
from reference_data.catalogs import CatalogSnapshot
from reference_data.instruments import InstrumentId
from runtime import ImplementationIdentity, RuntimeFiles
from serialization import DomainCodec

publication = DatasetPublication(SourceAdapters(), ImplementationIdentity.sha256())

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
assert (root / "configs/reference/binance-spot.json").is_file(), (
    "Run from the repository root or examples/."
)
notebook_path = Path(
    os.environ.get(
        "ALPHA_RESEARCH_NOTEBOOK_PATH", root / "examples/foundation-contract-checks.ipynb"
    )
).resolve()


class NotebookDisplay:
    """Render small tables for the exploratory notebook scenarios."""

    @staticmethod
    def show_table(headers: tuple[str, ...], rows: list[tuple[object, ...]]) -> None:
        """Display a small escaped table without adding a dataframe dependency.

        Args:
            headers: Column labels.
            rows: Exact or presentation values to render as text.
        """
        heading = "".join(
            f'<th style="text-align:left;padding:6px">{escape(label)}</th>' for label in headers
        )
        body = "".join(
            "<tr>"
            + "".join(
                f'<td style="padding:6px;vertical-align:top">{escape(str(value))}</td>'
                for value in row
            )
            + "</tr>"
            for row in rows
        )
        display(HTML(f"<table><thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table>"))

## 2. Which assumption will I change?


Edit this cell for a new synthetic question. Change one factor at a time initially.
`exact_microsecond` changes capture precision; `smaller_quantity` changes trade size;
`unconfigured_symbol` explores a rejected source message. Add your own named cases.
The fixed regression examples in section 7 do not depend on these parameters.

In [2]:
QUESTION = (
    "How do capture precision, quantity and symbol selection affect "
    "decoded availability and exact notional?"
)
HYPOTHESIS = (
    "Submicrosecond capture is rounded upward, quantity changes notional exactly, "
    "and an unknown symbol is rejected."
)
EXPERIMENT_ID = "foundation-capture-and-units-v1"
SEED = 0  # Recorded for identity; this experiment uses no randomness.

INSTRUMENT_ID = "11111111-1111-4111-8111-111111111111"
SAMPLE_START = datetime(2024, 6, 1, tzinfo=UTC)
SAMPLE_END = SAMPLE_START + timedelta(hours=1)
LOCAL_RECEIVED = datetime(2026, 9, 11, tzinfo=UTC)
LOCAL_INGESTED = LOCAL_RECEIVED + timedelta(seconds=1)
EVENT_TIME_MS = 1717199999993
FOCUS_SCENARIO = "reference"
CUTOFF_OFFSETS_US = (-1, 0, 1, 1000000)

SCENARIOS = (
    {
        "name": "reference",
        "capture": "2024-06-01T00:00:00.0041023Z",
        "symbol": "BTCUSDT",
        "price": "67540.01000000",
        "quantity": "0.00146000",
    },
    {
        "name": "exact_microsecond",
        "capture": "2024-06-01T00:00:00.0041020Z",
        "symbol": "BTCUSDT",
        "price": "67540.01000000",
        "quantity": "0.00146000",
    },
    {
        "name": "smaller_quantity",
        "capture": "2024-06-01T00:00:00.0041023Z",
        "symbol": "BTCUSDT",
        "price": "67540.01000000",
        "quantity": "0.00100000",
    },
    {
        "name": "unconfigured_symbol",
        "capture": "2024-06-01T00:00:00.0041023Z",
        "symbol": "SOLUSDT",
        "price": "100.00",
        "quantity": "1.00",
    },
)

display(Markdown(f"**Question:** {QUESTION}\n\n**Expected mechanism:** {HYPOTHESIS}"))
NotebookDisplay.show_table(
    ("Scenario", "Capture text", "Symbol", "Price", "Quantity"),
    [
        (item["name"], item["capture"], item["symbol"], item["price"], item["quantity"])
        for item in SCENARIOS
    ],
)

**Question:** How do capture precision, quantity and symbol selection affect decoded availability and exact notional?

**Expected mechanism:** Submicrosecond capture is rounded upward, quantity changes notional exactly, and an unknown symbol is rejected.

Scenario,Capture text,Symbol,Price,Quantity
reference,2024-06-01T00:00:00.0041023Z,BTCUSDT,67540.01000000,0.00146000
exact_microsecond,2024-06-01T00:00:00.0041020Z,BTCUSDT,67540.01000000,0.00146000
smaller_quantity,2024-06-01T00:00:00.0041023Z,BTCUSDT,67540.01000000,0.00100000
unconfigured_symbol,2024-06-01T00:00:00.0041023Z,SOLUSDT,100.00,1.00


## 3. Can someone identify the same code and inputs?


These checks compare installed runtime bytes with the source tree. If they fail,
refresh the frozen non-editable install and restart the kernel. A package path
alone is insufficient evidence that an edit is being exercised.

The runtime/input diff digest covers source, reference configuration and environment
inputs relative to HEAD. The notebook source digest excludes outputs, so refreshing
tables does not change the experiment's code identity. Successful runner snapshots
retain this table alongside inputs and findings; failed snapshots retain every cell
reached before the failure. These are local records, not the future formal trial store.

In [3]:
assert platform.python_version() == (root / ".python-version").read_text().strip() == "3.14.7"
installed = Path(market_data.__file__).resolve().parent.parent
assert "site-packages" in installed.parts
source_root = root / "src"
installed_inputs = dict(RuntimeFiles.inputs())
runtime_files = sorted(
    path for package in RuntimeFiles.roots for path in (source_root / package).rglob("*")
    if path.is_file() and (path.suffix in {".py", ".json"} or path.name == "py.typed")
)
assert {path.relative_to(source_root).as_posix() for path in runtime_files} == set(installed_inputs)
runtime_digest = hashlib.sha256()
for source_file in runtime_files:
    relative = source_file.relative_to(source_root)
    source_bytes = source_file.read_bytes()
    assert source_bytes == installed_inputs[relative.as_posix()].read_bytes(), f"Stale installed file: {relative}"
    runtime_digest.update(str(relative).encode() + b"\0" + source_bytes + b"\0")

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip()
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=root, text=True).strip()
working_status = subprocess.check_output(["git", "status", "--short"], cwd=root, text=True).strip()
candidate_diff = subprocess.check_output(
    [
        "git",
        "diff",
        "--binary",
        "HEAD",
        "--",
        "src",
        "configs",
        "pyproject.toml",
        "uv.lock",
        ".python-version",
    ],
    cwd=root,
)
notebook_document = json.loads(notebook_path.read_text())
notebook_source = [
    (cell["cell_type"], "".join(cell["source"])) for cell in notebook_document["cells"]
]
notebook_digest = hashlib.sha256(
    json.dumps(notebook_source, ensure_ascii=False, separators=(",", ":")).encode()
).hexdigest()
experiment_inputs = {
    "experiment_id": EXPERIMENT_ID,
    "question": QUESTION,
    "hypothesis": HYPOTHESIS,
    "seed": SEED,
    "instrument_id": INSTRUMENT_ID,
    "sample_start": SAMPLE_START.isoformat(),
    "sample_end": SAMPLE_END.isoformat(),
    "local_received": LOCAL_RECEIVED.isoformat(),
    "local_ingested": LOCAL_INGESTED.isoformat(),
    "event_time_ms": EVENT_TIME_MS,
    "scenarios": SCENARIOS,
    "focus_scenario": FOCUS_SCENARIO,
    "cutoff_offsets_us": CUTOFF_OFFSETS_US,
}
input_digest = hashlib.sha256(json.dumps(experiment_inputs, sort_keys=True).encode()).hexdigest()
identity = {
    "notebook_path": str(notebook_path),
    "attempt_id": os.environ.get("ALPHA_RESEARCH_ATTEMPT_ID", "interactive: unsnapshotted"),
    "commit": commit,
    "branch": branch,
    "working_tree": "changed" if working_status else "clean",
    "python": platform.python_version(),
    "platform": platform.platform(),
    "kernel": sys.executable,
    "package": str(installed),
    "verified_runtime_files": len(runtime_files),
    "runtime_sha256": runtime_digest.hexdigest(),
    "runtime_input_diff_sha256": hashlib.sha256(candidate_diff).hexdigest(),
    "uv_lock_sha256": hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest(),
    "catalog_file_sha256": hashlib.sha256(
        (root / "configs/reference/binance-spot.json").read_bytes()
    ).hexdigest(),
    "notebook_source_sha256": notebook_digest,
    "experiment_inputs_sha256": input_digest,
    "notebook_tooling": {
        item.metadata["Name"]: item.version
        for item in distributions()
        if item.metadata["Name"] in {"nbclient", "nbformat", "ipykernel"}
    },
}
NotebookDisplay.show_table(("Identity", "Observed value"), list(identity.items()))

Identity,Observed value
notebook_path,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/examples/foundation-contract-checks.ipynb
attempt_id,bfac2d07-4ac3-47c1-8878-4422cc191f05
commit,df2831e3e048d3d2a512690e7e010e9acd2df788
branch,codex/mr-08-saved-archive-rows
working_tree,changed
python,3.14.7
platform,macOS-27.0-arm64-arm-64bit-Mach-O
kernel,/Users/lxg/.cache/uv/builds-v0/.tmp8lKp71/bin/python
package,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/.venv/lib/python3.14/site-packages/alpha_research
verified_runtime_files,59


## 4. How do time precision, quantity and instrument choice affect the result?


Read the authored catalog and select the canonical instrument explicitly. Its
validity bounds are engineering metadata, not historical venue eligibility evidence.
Each scenario below calls the existing decoder; there is no notebook implementation
of timestamp parsing or money arithmetic.

In [4]:
domain_codec = DomainCodec()
catalog = domain_codec.decode_json(
    (root / "configs/reference/binance-spot.json").read_bytes(), CatalogSnapshot
)
instrument = catalog.resolve(
    InstrumentId.from_canonical_text(INSTRUMENT_ID), SAMPLE_START, SAMPLE_END
)
decoder = TardisDecoder(
    payload_decoder=BinancePayloadDecoder((instrument,)),
    source_id="tardis.binance.spot",
)
NotebookDisplay.show_table(
    ("Reference field", "Value"),
    [
        ("Instrument", instrument.symbol),
        ("Canonical ID", instrument.instrument_id),
        ("Currency", instrument.quote_currency),
        ("Tick", instrument.price_tick),
        ("Valid interval", f"[{instrument.valid_from}, {instrument.valid_to})"),
        ("Catalog identity", catalog.identity),
        ("Evidence", catalog.evidence),
    ],
)

events: dict[str, Trade] = {}
scenario_outcomes: list[tuple[object, ...]] = []
for sequence, scenario in enumerate(SCENARIOS):
    if scenario["name"] in {row[0] for row in scenario_outcomes}:
        raise ValueError("Scenario names must be unique")
    payload = {
        "stream": f"{scenario['symbol'].lower()}@trade",
        "data": {
            "e": "trade",
            "E": EVENT_TIME_MS + 1,
            "s": scenario["symbol"],
            "t": 7 + sequence,
            "p": scenario["price"],
            "q": scenario["quantity"],
            "T": EVENT_TIME_MS,
            "m": False,
        },
    }
    line = scenario["capture"] + " " + json.dumps(payload, separators=(",", ":"))
    try:
        event = decoder.decode_line(
            line,
            source_sequence=sequence,
            local_received=LOCAL_RECEIVED,
            local_ingested=LOCAL_INGESTED,
        )
    except MarketEventError as error:
        scenario_outcomes.append((scenario["name"], "rejected", str(error)))
    else:
        assert isinstance(event, Trade)
        events[scenario["name"]] = event
        scenario_outcomes.append((scenario["name"], "decoded", event.provenance.event_id))
NotebookDisplay.show_table(("Scenario", "Outcome", "Identity / reason"), scenario_outcomes)

dimension = Dimension(instrument_id=instrument.instrument_id, kind=QuantityKind.BASE_UNIT)
notionals: dict[str, Money] = {}
decoded_rows: list[tuple[object, ...]] = []
for name, event in events.items():
    notional = Price(value=event.price, currency=instrument.quote_currency, per=dimension).amount(
        Quantity(value=event.quantity, dimension=dimension)
    )
    notionals[name] = notional
    decoded_rows.append(
        (
            name,
            event.original_capture_time,
            event.provenance.ts_knowledge,
            event.price,
            event.quantity,
            notional.value,
            notional.currency.code,
        )
    )
NotebookDisplay.show_table(
    (
        "Scenario",
        "Source capture",
        "Knowledge UTC",
        "Price",
        "Quantity",
        "Exact notional",
        "Currency",
    ),
    decoded_rows,
)

Reference field,Value
Instrument,BTCUSDT
Canonical ID,11111111-1111-4111-8111-111111111111
Currency,code='USDT' minor_unit=Decimal('1E-8')
Tick,0.01
Valid interval,"[2024-06-01 00:00:00+00:00, 2024-06-02 00:00:00+00:00)"
Catalog identity,3125723d3760b4a35238b75a493e1d1e2a6956c66566ac7231c35a0afb59bad6
Evidence,Authored local research assumptions for the bounded Tardis Binance spot sample; this catalog does not assert historical exchange filters or point-in-time venue truth.


Scenario,Outcome,Identity / reason
reference,decoded,tardis.binance.spot:0:trade:7
exact_microsecond,decoded,tardis.binance.spot:1:trade:8
smaller_quantity,decoded,tardis.binance.spot:2:trade:9
unconfigured_symbol,rejected,unsupported stream symbol SOLUSDT


Scenario,Source capture,Knowledge UTC,Price,Quantity,Exact notional,Currency
reference,2024-06-01T00:00:00.0041023Z,2024-06-01 00:00:00.004103+00:00,67540.01000000,0.00146000,98.608414600000000000,USDT
exact_microsecond,2024-06-01T00:00:00.0041020Z,2024-06-01 00:00:00.004102+00:00,67540.01000000,0.00146000,98.608414600000000000,USDT
smaller_quantity,2024-06-01T00:00:00.0041023Z,2024-06-01 00:00:00.004103+00:00,67540.01000000,0.00100000,67.540010000000000000,USDT


## 5. Was this observation available at my decision time?


Change `FOCUS_SCENARIO` and `CUTOFF_OFFSETS_US` in the inputs cell. The two predicates
below ask different questions: is this assertion usable for its original business
instant, and is it valid at the decision instant itself? Point events have a
one-microsecond validity interval; this API is not a last-observation selector.

In [5]:
focus = events[FOCUS_SCENARIO]
provenance = focus.provenance
NotebookDisplay.show_table(
    ("Clock", "Observed UTC instant"),
    [
        ("Source event", provenance.ts_event),
        ("Source received", provenance.ts_source_received),
        ("Knowledge", provenance.ts_knowledge),
        ("Local received", provenance.ts_received),
        ("Local ingested", provenance.ts_ingested),
        ("Valid from", provenance.valid_from),
        ("Valid to (exclusive)", provenance.valid_to),
    ],
)
availability_rows: list[tuple[object, ...]] = []
for offset in CUTOFF_OFFSETS_US:
    cutoff = provenance.ts_knowledge + timedelta(microseconds=offset)
    availability_rows.append(
        (
            offset,
            cutoff,
            provenance.is_available(cutoff, provenance.valid_from),
            provenance.is_available(cutoff, cutoff),
        )
    )
NotebookDisplay.show_table(
    ("Cutoff offset (us)", "Decision cutoff", "Valid at original point?", "Valid at decision?"),
    availability_rows,
)

Clock,Observed UTC instant
Source event,2024-05-31 23:59:59.993000+00:00
Source received,2024-06-01 00:00:00.004103+00:00
Knowledge,2024-06-01 00:00:00.004103+00:00
Local received,2026-09-11 00:00:00+00:00
Local ingested,2026-09-11 00:00:01+00:00
Valid from,2024-06-01 00:00:00.004103+00:00
Valid to (exclusive),2024-06-01 00:00:00.004104+00:00


Cutoff offset (us),Decision cutoff,Valid at original point?,Valid at decision?
-1,2024-06-01 00:00:00.004102+00:00,False,False
0,2024-06-01 00:00:00.004103+00:00,True,True
1,2024-06-01 00:00:00.004104+00:00,True,False
1000000,2024-06-01 00:00:01.004103+00:00,True,False


## 6. What happens when the observation is missing or the instrument is wrong?


Inspect a blank-line disconnect, a request that exceeds catalog coverage, and a
quantity for the wrong instrument. These exercise existing package boundaries.
The next-capture proxy remains a separate source-reader responsibility; this
notebook supplies the explicit interval-end proxy.

In [6]:
disconnect = decoder.decode_line(
    "",
    source_sequence=len(SCENARIOS),
    local_received=LOCAL_RECEIVED,
    local_ingested=LOCAL_INGESTED,
    disconnect_capture_time=SAMPLE_END,
    disconnect_capture_reason="interval_end",
)
assert isinstance(disconnect, Disconnect)
NotebookDisplay.show_table(
    ("Disconnect field", "Value"),
    [
        ("Proxy reason", disconnect.capture_proxy_reason),
        ("Knowledge", disconnect.provenance.ts_knowledge),
        ("Event", disconnect.provenance.ts_event),
        ("Instrument", disconnect.instrument_id),
        ("Source sequence", disconnect.provenance.source_sequence),
    ],
)
boundary_outcomes: list[tuple[object, ...]] = []
try:
    catalog.resolve(
        instrument.instrument_id,
        instrument.valid_from,
        instrument.valid_to + timedelta(microseconds=1),
    )
except ValueError as error:
    boundary_outcomes.append(("Catalog end + 1 microsecond", type(error).__name__, str(error)))
else:
    raise AssertionError("Incomplete catalog coverage was unexpectedly accepted")
other_dimension = Dimension(
    instrument_id=InstrumentId(UUID("33333333-3333-4333-8333-333333333333")),
    kind=QuantityKind.BASE_UNIT,
)
try:
    Price(value=focus.price, currency=instrument.quote_currency, per=dimension).amount(
        Quantity(value=Decimal("1"), dimension=other_dimension)
    )
except UnitError as error:
    boundary_outcomes.append(("Wrong quantity instrument", type(error).__name__, str(error)))
else:
    raise AssertionError("Mismatched quantity units were unexpectedly accepted")
NotebookDisplay.show_table(("Boundary", "Observed exception", "Reason"), boundary_outcomes)

Disconnect field,Value
Proxy reason,interval_end
Knowledge,2024-06-01 01:00:00+00:00
Event,2024-06-01 01:00:00+00:00
Instrument,None
Source sequence,4


Boundary,Observed exception,Reason
Catalog end + 1 microsecond,ValueError,instrument does not cover the complete requested interval
Wrong quantity instrument,UnitError,price and quantity dimension mismatch


## 7. Can I reconcile a small example by hand?


Keep these constants fixed. They retain the original MR 00 clock, notional and
hand-ledger checks. Add alternate economics to a new exploratory section rather
than changing the expected golden result. These are reference calculations, not
a packaged fee engine or account ledger.

In [7]:
reference_start = datetime(2024, 6, 1, tzinfo=UTC)
reference_instrument = catalog.resolve(
    InstrumentId.from_canonical_text("11111111-1111-4111-8111-111111111111"),
    reference_start,
    reference_start + timedelta(hours=1),
)
reference_line = (
    "2024-06-01T00:00:00.0041023Z "
    '{"stream":"btcusdt@trade","data":{"e":"trade","E":1717199999994,'
    '"s":"BTCUSDT","t":7,"p":"67540.01000000","q":"0.00146000",'
    '"T":1717199999993,"m":false}}'
)
reference_received = datetime(2026, 9, 11, tzinfo=UTC)
reference_decoder = TardisDecoder(
    payload_decoder=BinancePayloadDecoder((reference_instrument,)),
    source_id="tardis.binance.spot",
)
reference_event = reference_decoder.decode_line(
    reference_line,
    source_sequence=0,
    local_received=reference_received,
    local_ingested=reference_received + timedelta(seconds=1),
    source_digest="a" * 64,
)
assert isinstance(reference_event, Trade)
reference_capture = reference_start + timedelta(microseconds=4103)
reference_provenance = reference_event.provenance
reference_dimension = Dimension(
    instrument_id=reference_instrument.instrument_id, kind=QuantityKind.BASE_UNIT
)
reference_notional = Price(
    value=reference_event.price,
    currency=reference_instrument.quote_currency,
    per=reference_dimension,
).amount(Quantity(value=reference_event.quantity, dimension=reference_dimension))
reference_checks = [
    ("Exchange clock", reference_provenance.ts_event, reference_start - timedelta(milliseconds=7)),
    ("Capture ceiling", reference_provenance.ts_source_received, reference_capture),
    ("Knowledge clock", reference_provenance.ts_knowledge, reference_capture),
    ("Local receipt", reference_provenance.ts_received, reference_received),
    (
        "Local ingestion",
        reference_provenance.ts_ingested,
        reference_received + timedelta(seconds=1),
    ),
    (
        "Point endpoint",
        reference_provenance.valid_to,
        reference_capture + timedelta(microseconds=1),
    ),
    ("Exact notional", reference_notional.value, Decimal("98.60841460")),
    ("USDT quantum", reference_notional.currency.minor_unit, Decimal("0.00000001")),
]
NotebookDisplay.show_table(
    ("Reference check", "Actual", "Expected", "Pass"),
    [(name, actual, expected, actual == expected) for name, actual, expected in reference_checks],
)
assert all(actual == expected for _, actual, expected in reference_checks)
assert reference_provenance.is_available(reference_capture, reference_capture)
assert not reference_provenance.is_available(
    reference_capture - timedelta(microseconds=1), reference_capture
)
assert not reference_provenance.is_available(
    reference_capture + timedelta(hours=1), reference_capture + timedelta(hours=1)
)

Reference check,Actual,Expected,Pass
Exchange clock,2024-05-31 23:59:59.993000+00:00,2024-05-31 23:59:59.993000+00:00,True
Capture ceiling,2024-06-01 00:00:00.004103+00:00,2024-06-01 00:00:00.004103+00:00,True
Knowledge clock,2024-06-01 00:00:00.004103+00:00,2024-06-01 00:00:00.004103+00:00,True
Local receipt,2026-09-11 00:00:00+00:00,2026-09-11 00:00:00+00:00,True
Local ingestion,2026-09-11 00:00:01+00:00,2026-09-11 00:00:01+00:00,True
Point endpoint,2024-06-01 00:00:00.004104+00:00,2024-06-01 00:00:00.004104+00:00,True
Exact notional,98.608414600000000000,98.60841460,True
USDT quantum,1E-8,1E-8,True


### Check cash and fees with an independent hand ledger


The arbitrary QXX currency has quantum `0.01`; fees use explicit Decimal half-up
rounding. This preserves the roadmap oracle. The selected P1 baseline instead
uses USDT quantum `0.00000001` and C03 half-even rounding. Neither choice here
claims a venue fee schedule. The reference instrument is deliberately arbitrary.

In [8]:
oracle_currency = Currency(code="QXX", minor_unit=Decimal("0.01"))
oracle_dimension = Dimension(
    instrument_id=InstrumentId(UUID("33333333-3333-4333-8333-333333333333")),
    kind=QuantityKind.BASE_UNIT,
)
oracle_quantity = Quantity(value=Decimal("2"), dimension=oracle_dimension)
buy = Price(value=Decimal("100.50"), currency=oracle_currency, per=oracle_dimension).amount(
    oracle_quantity
)
sell = Price(value=Decimal("109.50"), currency=oracle_currency, per=oracle_dimension).amount(
    oracle_quantity
)
buy_fee = Money(
    value=(buy.value * Decimal("0.001")).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP),
    currency=oracle_currency,
)
sell_fee = Money(
    value=(sell.value * Decimal("0.001")).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP),
    currency=oracle_currency,
)
initial = Money(value=Decimal("1000.00"), currency=oracle_currency)
cash_after_buy = initial - buy - buy_fee
open_mark = Price(value=Decimal("105.00"), currency=oracle_currency, per=oracle_dimension).amount(
    oracle_quantity
)
final_cash = cash_after_buy + sell - sell_fee
oracle_checks = [
    ("Buy fee", buy_fee.value, Decimal("0.20")),
    ("Sell fee", sell_fee.value, Decimal("0.22")),
    ("Cash after buy", cash_after_buy.value, Decimal("798.80")),
    ("Open NAV", (cash_after_buy + open_mark).value, Decimal("1008.80")),
    ("Final cash", final_cash.value, Decimal("1017.58")),
    ("Net P&L", (final_cash - initial).value, Decimal("17.58")),
]
NotebookDisplay.show_table(
    ("Golden", "Actual", "Expected", "Pass"),
    [(name, actual, expected, actual == expected) for name, actual, expected in oracle_checks],
)
assert all(actual == expected for _, actual, expected in oracle_checks)
assert (final_cash - initial).value == Decimal("20.00") - Decimal("2.00") - Decimal("0.42")
NotebookDisplay.show_table(
    ("Ledger step", "Cash change", "Cash", "Position", "NAV"),
    [
        ("Initial / no-trade", Decimal("0"), initial.value, Decimal("0"), initial.value),
        (
            "Buy; mark at 105",
            -(buy + buy_fee).value,
            cash_after_buy.value,
            Decimal("2"),
            (cash_after_buy + open_mark).value,
        ),
        ("Sell", (sell - sell_fee).value, final_cash.value, Decimal("0"), final_cash.value),
    ],
)

Golden,Actual,Expected,Pass
Buy fee,0.20,0.20,True
Sell fee,0.22,0.22,True
Cash after buy,798.800000000000000000,798.80,True
Open NAV,1008.800000000000000000,1008.80,True
Final cash,1017.580000000000000000,1017.58,True
Net P&L,17.580000000000000000,17.58,True


Ledger step,Cash change,Cash,Position,NAV
Initial / no-trade,0,1000.00,0,1000.00
Buy; mark at 105,-201.200000000000000000,798.800000000000000000,2,1008.800000000000000000
Sell,218.780000000000000000,1017.580000000000000000,0,1017.580000000000000000


## 8. What did this experiment establish?


The computed summary describes this execution. Edit the interpretation and next
question below as ideas develop. Keep a new experiment ID when changing the
question or scenario set. Run through the runner to preserve a distinct executed
snapshot before replacing conclusions; direct interactive edits are not archived
automatically. Formal research still needs the retained trial fields and protected
input/output references; ignored local snapshots are not a durable research service.

In [9]:
findings = [
    ("Scenario results", f"{len(events)} decoded; {len(SCENARIOS) - len(events)} rejected"),
    ("Focused scenario", FOCUS_SCENARIO),
    ("Focused notional", f"{notionals[FOCUS_SCENARIO].value} {instrument.quote_currency.code}"),
    ("Observed rejection boundaries", len(boundary_outcomes)),
    ("Fixed clock/unit references", f"{len(reference_checks)} passed"),
    (
        "Fixed ledger references",
        f"{len(oracle_checks)} passed; net {(final_cash - initial).value} QXX",
    ),
    (
        "Current external evidence",
        "Synthetic Tardis foundation; actual data workflow is in research-foundations.ipynb",
    ),
]
NotebookDisplay.show_table(("Finding", "Evidence in this execution"), findings)

INTERPRETATION = (
    "Exact arithmetic and explicit temporal boundaries support controlled synthetic exploration."
)
NEXT_QUESTION = (
    "Which additional input boundary would improve understanding "
    "before implementing bounded source requests?"
)
PROMOTION_DECISION = (
    "Reuse the existing decoder, provenance and financial APIs. "
    "No new runtime implementation is needed for this experiment."
)
NotebookDisplay.show_table(
    ("Research note", "Current decision"),
    [
        ("Interpretation", INTERPRETATION),
        ("Next question", NEXT_QUESTION),
        ("Promotion", PROMOTION_DECISION),
    ],
)

Finding,Evidence in this execution
Scenario results,3 decoded; 1 rejected
Focused scenario,reference
Focused notional,98.608414600000000000 USDT
Observed rejection boundaries,2
Fixed clock/unit references,8 passed
Fixed ledger references,6 passed; net 17.580000000000000000 QXX
Current external evidence,Synthetic Tardis foundation; actual data workflow is in research-foundations.ipynb


Research note,Current decision
Interpretation,Exact arithmetic and explicit temporal boundaries support controlled synthetic exploration.
Next question,Which additional input boundary would improve understanding before implementing bounded source requests?
Promotion,"Reuse the existing decoder, provenance and financial APIs. No new runtime implementation is needed for this experiment."


## 9. Choose the next research activity


| Working content | Eventual notebook | Promotion boundary |
| --- | --- | --- |
| Permitted requests, source messages and acquisition inspection | Data fetching | Stable acquisition, validation and storage APIs |
| Saved sample inspection, coverage, distributions and features | EDA | Shared transforms with retained consumers |
| Signals, economics, strategy comparisons and attribution | Strategy research | Selected versioned strategy and simulation/accounting implementations |
| Versioned reports and actual runtime observations | Strategy monitoring | Persistent metrics, alerts and operational controls |

These notebooks will emerge as working inputs and capabilities exist. The fixed
checks here remain backed by tracked tests; they do not require every exploratory
idea to become production code. Package APIs should own promoted behavior, and
notebooks should then call those implementations for formal results.

## 10. Am I selecting the intended instruments and intervals? (NB 01–02)


Question: can a named universe select either supported spot instrument through the
same installed implementation, with exact minute partitions and explicit failures?
These are authored synthetic frames. The frozen P1 universe and source-access
status are unchanged. Configuration selects only fixture inspection.

The following cells consume the installed configuration, request and pipeline APIs;
they do not recreate decoders or request partitioning. Inspect both single-instrument
jobs and a two-instrument job, then run the installed command on saved bytes.

In [10]:
# third party library imports
from msgspec.structs import replace

# project library imports
from applications.inspect_fixture import inspect_fixture
from configuration import Configuration
from market_data.contracts import SourceRequest

fixture_inspection_config_path = root / "configs/data-jobs/fixture-inspection.json"
fixture_inspection_fixture_root = root / "tests/fixtures/market_data/tardis"
fixture_inspection_configuration = Configuration.read(fixture_inspection_config_path)
fixture_inspection_rows = []
fixture_inspection_manifests = {}
for job_name in ("btc-fixture", "eth-fixture", "both-fixture"):
    manifest = fixture_inspection_configuration.resolve(job_name, catalog, implementation_identity=ImplementationIdentity.sha256())
    requests = SourceRequest.partition(manifest)
    fixture_inspection_manifests[job_name] = manifest
    assert len(requests) == 60 * len(manifest.instruments)
    assert requests[0].start == manifest.job.start
    assert requests[-1].end == manifest.job.end
    fixture_inspection_rows.append(
        (
            job_name,
            ", ".join(i.symbol for i in manifest.instruments),
            len(requests),
            requests[0].start,
            requests[-1].end,
            requests[-1].offset_minutes,
            manifest.identity,
        )
    )
NotebookDisplay.show_table(
    (
        "Job",
        "Selected symbols",
        "Requests",
        "First start UTC",
        "Last end UTC",
        "Last minute offset",
        "Run identity",
    ),
    fixture_inspection_rows,
)
NotebookDisplay.show_table(
    ("Offset", "Symbol", "Start UTC", "End UTC", "Request identity"),
    [
        (r.offset_minutes, r.instrument.symbol, r.start, r.end, r.identity)
        for r in (*requests[:4], *requests[-2:])
    ],
)

fixture_inspection_event_rows = []
for job_name, symbol in (("btc-fixture", "btcusdt"), ("eth-fixture", "ethusdt")):
    request, event, digest = inspect_fixture(
        fixture_inspection_manifests[job_name],
        fixture_inspection_fixture_root / f"{symbol}.txt",
        local_received=datetime(2026, 9, 14, tzinfo=UTC),
        local_ingested=datetime(2026, 9, 14, 0, 0, 1, tzinfo=UTC),
        source_sequence=0,
    )
    assert event.instrument_id == request.instrument.instrument_id
    fixture_inspection_event_rows.append(
        (
            job_name,
            event.symbol,
            event.price,
            event.provenance.ts_event,
            event.provenance.ts_knowledge,
            event.provenance.ts_received,
            digest,
        )
    )
NotebookDisplay.show_table(
    (
        "Job",
        "Event symbol",
        "Exact price",
        "Source event UTC",
        "Capture knowledge UTC",
        "Original receipt UTC",
        "Fixture SHA-256",
    ),
    fixture_inspection_event_rows,
)

Job,Selected symbols,Requests,First start UTC,Last end UTC,Last minute offset,Run identity
btc-fixture,BTCUSDT,60,2024-06-01 00:00:00+00:00,2024-06-01 01:00:00+00:00,59,6bbafd9eb42948947f92515b8209cd95cd2049c0f7061ce8a6a4dc343a8b88c8
eth-fixture,ETHUSDT,60,2024-06-01 00:00:00+00:00,2024-06-01 01:00:00+00:00,59,5c8803a188255012530e808e74289ce7aea5bdac6625d32665201dfebe3bc464
both-fixture,"BTCUSDT, ETHUSDT",120,2024-06-01 00:00:00+00:00,2024-06-01 01:00:00+00:00,59,c69a2037584c948fd4649e0364b9a515499c11b3a3f302ba9f22d4847abb27cd


Offset,Symbol,Start UTC,End UTC,Request identity
0,BTCUSDT,2024-06-01 00:00:00+00:00,2024-06-01 00:01:00+00:00,ac36dc508dbb6f064d22400f0ac23f3ee961aec43586081886d2600399ead59a
0,ETHUSDT,2024-06-01 00:00:00+00:00,2024-06-01 00:01:00+00:00,7fcfd4025939fd4d508731d6967888e5c968d3f895533e0c28aca453a2cbbfa2
1,BTCUSDT,2024-06-01 00:01:00+00:00,2024-06-01 00:02:00+00:00,c21c02aac7f88ea1981bc5406ca6c5c5c9411a8aee250153bd778bd3acebe51f
1,ETHUSDT,2024-06-01 00:01:00+00:00,2024-06-01 00:02:00+00:00,8d90817b406c498593c770f598048f7d90b27477e957c2bfe517f3a0b125ddd9
59,BTCUSDT,2024-06-01 00:59:00+00:00,2024-06-01 01:00:00+00:00,8da9d2dd63733ba06fe19397a7069b81e4bd1d7a6fca0e7209f3739fcb67c18a
59,ETHUSDT,2024-06-01 00:59:00+00:00,2024-06-01 01:00:00+00:00,5619b566c341d86ba91c1cec86ce85a05aac82a0e67264ac1b69dbbcb5d82762


Job,Event symbol,Exact price,Source event UTC,Capture knowledge UTC,Original receipt UTC,Fixture SHA-256
btc-fixture,BTCUSDT,67540.01000000,2024-05-31 23:59:59.993000+00:00,2024-06-01 00:00:00.004103+00:00,2026-09-14 00:00:00+00:00,30a770208c525b39a5a9f8e9eb4f21b89bdb4b48bf063d6007a9174ed1b75915
eth-fixture,ETHUSDT,3800.01000000,2024-05-31 23:59:59.993000+00:00,2024-06-01 00:00:00.004103+00:00,2026-09-14 00:00:00+00:00,af186c0065aab0a4a841ef9cf447bb0f7ba2628fb6428b1367a4ebba8d7fbf87


In [11]:
fixture_inspection_failures = []
try:
    domain_codec.decode_json(
        fixture_inspection_config_path.read_text().replace(
            '"product": "spot"', '"product": "future"', 1
        ),
        Configuration,
    )
except ValueError as error:
    fixture_inspection_failures.append(("Unsupported family", str(error)))
else:
    raise AssertionError("unsupported product was accepted")
try:
    shortened = replace(
        catalog,
        instruments=tuple(
            replace(i, valid_to=i.valid_from + timedelta(minutes=59)) for i in catalog.instruments
        ),
    )
    fixture_inspection_configuration.resolve("btc-fixture", shortened, implementation_identity=ImplementationIdentity.sha256())
except ValueError as error:
    fixture_inspection_failures.append(("Incomplete catalog interval", str(error)))
else:
    raise AssertionError("incomplete catalog was accepted")
NotebookDisplay.show_table(("Rejected scenario", "Diagnostic"), fixture_inspection_failures)

# standard library imports
import shutil

command_path = shutil.which("alpha-inspect-fixture")
assert command_path is not None, "Refresh the installed fixture command"
command = Path(command_path)
cli_rows = []
for job_name, filename, expected in (
    ("btc-fixture", "btcusdt.txt", 0),
    ("eth-fixture", "ethusdt.txt", 0),
    ("missing-binding", "btcusdt.txt", 2),
    ("btc-fixture", "ethusdt.txt", 3),
    ("btc-fixture", "invalid-timestamp.txt", 3),
):
    completed = subprocess.run(
        [
            str(command),
            "--config",
            str(fixture_inspection_config_path),
            "--catalog",
            str(root / "configs/reference/binance-spot.json"),
            "--job",
            job_name,
            "--fixture",
            str(fixture_inspection_fixture_root / filename),
            "--received",
            "2026-09-14T00:00:00Z",
            "--ingested",
            "2026-09-14T00:00:01Z",
            "--sequence",
            "0",
        ],
        capture_output=True,
        text=True,
        check=False,
    )
    assert completed.returncode == expected, completed.stderr
    report = json.loads(completed.stdout if expected == 0 else completed.stderr)
    cli_rows.append(
        (
            job_name,
            filename,
            expected,
            report["event"]["symbol"] if expected == 0 else report["diagnostic"],
            report["run_identity"],
        )
    )
NotebookDisplay.show_table(
    ("Job", "Saved frame", "Exit", "Result/diagnostic", "Run identity"), cli_rows
)
NotebookDisplay.show_table(
    ("M0 input", "SHA-256"),
    [
        (str(path.relative_to(root)), hashlib.sha256(path.read_bytes()).hexdigest())
        for path in (
            fixture_inspection_config_path,
            *sorted(fixture_inspection_fixture_root.iterdir()),
        )
    ],
)

Rejected scenario,Diagnostic
Unsupported family,Invalid enum value 'future' - at `$.jobs[0].product`
Incomplete catalog interval,instrument does not cover the complete requested interval


Job,Saved frame,Exit,Result/diagnostic,Run identity
btc-fixture,btcusdt.txt,0,BTCUSDT,6bbafd9eb42948947f92515b8209cd95cd2049c0f7061ce8a6a4dc343a8b88c8
eth-fixture,ethusdt.txt,0,ETHUSDT,5c8803a188255012530e808e74289ce7aea5bdac6625d32665201dfebe3bc464
missing-binding,btcusdt.txt,2,unknown data-job binding: missing-binding,None
btc-fixture,ethusdt.txt,3,unsupported stream symbol ETHUSDT,6bbafd9eb42948947f92515b8209cd95cd2049c0f7061ce8a6a4dc343a8b88c8
btc-fixture,invalid-timestamp.txt,3,T must be nonnegative,6bbafd9eb42948947f92515b8209cd95cd2049c0f7061ce8a6a4dc343a8b88c8


M0 input,SHA-256
configs/data-jobs/fixture-inspection.json,cbbda41513b34530118982194372a1ea09e22d1800a37e7c654a6b3872b0b3ab
tests/fixtures/m0/btcusdt.txt,30a770208c525b39a5a9f8e9eb4f21b89bdb4b48bf063d6007a9174ed1b75915
tests/fixtures/m0/ethusdt.txt,af186c0065aab0a4a841ef9cf447bb0f7ba2628fb6428b1367a4ebba8d7fbf87
tests/fixtures/m0/invalid-timestamp.txt,59977c0f5ce69c851ac0e680281b65f20df4ad18d167c2ccd05c67fbc423e53b
tests/fixtures/m0/unsupported-product.json,3ec687324a567a041dd14364185ef458354d018c803161bedcf8dc1e70584b40


### Research interpretation

The same settings mechanism selects the intended instruments and time interval. Unsupported or mismatched input fails explicitly. These authored examples do not prove historical eligibility.

<details>
<summary>Detailed evidence and scope of this checkpoint</summary>

Both supported listings and the combined universe resolve through the same installed
code. Each selected hour yields exactly 60 minute requests per instrument. The
saved frames preserve exact prices, the preceding exchange event time and the
conservatively rounded capture time; capture time owns partition membership.
Unknown bindings, an unsupported family, incomplete catalog coverage, wrong
instruments and invalid exchange timestamps fail explicitly.

This is fixture evidence, with no provider access, historical eligibility proof,
raw-object store, dataset publication, or strategy execution. Section 11 now exercises MR 03's saved raw-object descriptor/verifier. The real Binance source workflow is in the historical notebook; this section remains an offline fixture check. The frozen experiment and its hand ledger are unchanged.

</details>

## 11. Has my saved input changed? (NB 03)

**Read the result:** Changing even one saved byte must break the original file identity. A successful read establishes the retained bytes, not provider completeness.

<details>
<summary>Implementation and checkpoint details</summary>

Question: does the saved object still contain the exact bytes assigned to this
request partition? The installed `RawObject` records SHA-256, byte size, the
existing request identity, a synthetic retrieval reference and original local
receipt/ingestion times. It does not infer source validity or replace the clocks
with verification time. The factory and verifier read in bounded binary chunks.

The sample below compresses the retained synthetic BTC frame, verifies and
round-trips its descriptor, then changes one gzip header byte. Decompression still
yields the same frame, but byte identity must fail. Truncation and a descriptor
used for another minute must also fail. All files live in a temporary directory.

</details>

In [12]:
# standard library imports
import gzip
from tempfile import TemporaryDirectory

# project library imports
from market_data.contracts import RawObject

raw_request = SourceRequest.partition(fixture_inspection_manifests["btc-fixture"])[0]
raw_payload = (fixture_inspection_fixture_root / "btcusdt.txt").read_bytes()
raw_compressed = gzip.compress(raw_payload, mtime=0)
raw_failures = []
with TemporaryDirectory(prefix="raw-object-notebook-") as raw_directory:
    raw_path = Path(raw_directory) / "fixture.gz"
    raw_path.write_bytes(raw_compressed)
    raw_descriptor = RawFiles.describe(
        raw_path,
        request=raw_request.binding,
        retrieval_reference="synthetic:m0-btc-gzip-v1",
        ts_received=datetime(2026, 9, 14, tzinfo=UTC),
        ts_ingested=datetime(2026, 9, 14, 0, 0, 1, tzinfo=UTC),
    )
    raw_wire = domain_codec.encode_json(raw_descriptor)
    raw_restored = domain_codec.decode_json(raw_wire, RawObject)
    assert raw_restored == raw_descriptor
    assert RawFiles.verify(raw_restored, raw_path, request=raw_request.binding) is raw_restored
    assert raw_descriptor.sha256 == hashlib.sha256(raw_compressed).hexdigest()
    assert raw_descriptor.sha256 != hashlib.sha256(raw_payload).hexdigest()
    NotebookDisplay.show_table(
        ("Raw object field", "Verified value"),
        [
            ("Retained size (compressed bytes)", raw_descriptor.size_bytes),
            ("Retained SHA-256", raw_descriptor.sha256),
            ("Decompressed SHA-256 (different identity)", hashlib.sha256(raw_payload).hexdigest()),
            ("Request / partition identity", raw_descriptor.request_identity),
            ("Partition bounds [start, end)", f"{raw_request.start} / {raw_request.end}"),
            ("Retrieval reference", raw_descriptor.retrieval_reference),
            ("Original local receipt UTC", raw_descriptor.ts_received),
            ("Original local ingestion UTC", raw_descriptor.ts_ingested),
            ("Descriptor JSON round trip", raw_restored == raw_descriptor),
        ],
    )
    changed_header = raw_compressed[:4] + b"\x01" + raw_compressed[5:]
    assert gzip.decompress(changed_header) == raw_payload
    for label, payload, expected_request, message in (
        (
            "One changed header byte; same decoded content",
            changed_header,
            raw_request,
            "SHA-256 mismatch",
        ),
        ("Truncated retained bytes", raw_compressed[:-1], raw_request, "size mismatch"),
        (
            "Descriptor for another minute",
            raw_compressed,
            replace(raw_request, offset_minutes=1),
            "request identity mismatch",
        ),
    ):
        raw_path.write_bytes(payload)
        try:
            RawFiles.verify(raw_restored, raw_path, request=expected_request.binding)
        except ValueError as error:
            assert message in str(error)
            raw_failures.append((label, str(error)))
        else:
            raise AssertionError(f"Accepted invalid raw object: {label}")
    raw_path.write_bytes(raw_compressed)
    assert RawFiles.verify(raw_restored, raw_path, request=raw_request.binding) is raw_restored
    assert domain_codec.encode_json(raw_restored) == raw_wire
NotebookDisplay.show_table(("Rejected scenario", "Diagnostic"), raw_failures)
print(
    "NB 03: verified compressed bytes, retained clocks and JSON round trip; 3 expected failures; recovery passed."
)

Raw object field,Verified value
Retained size (compressed bytes),145
Retained SHA-256,e4a81e816955ccd1884b30986a6fea04c10b8c0eafd14233d0341cc6a0b19d26
Decompressed SHA-256 (different identity),30a770208c525b39a5a9f8e9eb4f21b89bdb4b48bf063d6007a9174ed1b75915
Request / partition identity,ac36dc508dbb6f064d22400f0ac23f3ee961aec43586081886d2600399ead59a
"Partition bounds [start, end)",2024-06-01 00:00:00+00:00 / 2024-06-01 00:01:00+00:00
Retrieval reference,synthetic:m0-btc-gzip-v1
Original local receipt UTC,2026-09-14 00:00:00+00:00
Original local ingestion UTC,2026-09-14 00:00:01+00:00
Descriptor JSON round trip,True


Rejected scenario,Diagnostic
One changed header byte; same decoded content,"raw object SHA-256 mismatch: expected e4a81e816955ccd1884b30986a6fea04c10b8c0eafd14233d0341cc6a0b19d26, got ecf46d7bdffc36f0101ac37041fe603c7063df5a494eac60ab9e0b30e8b038cd"
Truncated retained bytes,"raw object size mismatch: expected 145, got 144"
Descriptor for another minute,raw object request identity mismatch


NB 03: verified compressed bytes, retained clocks and JSON round trip; 3 expected failures; recovery passed.


### Research interpretation

A changed or truncated file cannot stand in for the original research input. Reopening the original still works and preserves its original timestamps. This demonstrates file identity, not complete source coverage.

<details>
<summary>Detailed evidence and scope of this checkpoint</summary>

Exact stored bytes own identity: even a gzip header change with identical decoded
content is rejected. A shorter file fails its byte count; the same bytes cannot
satisfy a descriptor assigned to a different request minute. Reopening after the
failures succeeds with the original bytes, descriptor and timestamps.

This verifies an existing local object against trusted metadata at read time.
The factory records the caller's attribution; it cannot authenticate a provider
or establish response completeness. Concurrent writers must be excluded by the
caller. No raw publication, acquisition, resume or dataset validity is claimed.
Section 12 exercises MR 04 publication separately. The frozen P1 experiment
remains unchanged.

</details>

## 12. Could an interrupted save leave me with half a file? (NB 04)

**Read the result:** Analysis must see either a complete original file or an explicit absence. A stopped writer must not leave a partly written file appearing ready.

<details>
<summary>Implementation and checkpoint details</summary>

Question: what can a reader observe when a publisher dies immediately before or
after atomic publication? `RawStore` accepts an existing storage directory and
stages on that filesystem. It verifies exact bytes, synchronizes the file, links
the digest name without overwriting a winner, then synchronizes the directory.
`open` checks the complete published object before returning a read-only stream.

Two isolated child processes use the retained synthetic interruption fixture and
exit without cleanup at the actual link boundary. Inspect the leftover staging
file and published name, retry through a new store instance, and confirm the
published inode is reused. Remove an orphan only after its writer has exited.
These are process-failure experiments, not physical power-loss tests.

</details>

In [13]:
# project library imports
from market_data.storage import RawStore

publication_rows = []
with TemporaryDirectory(prefix="raw-publication-notebook-") as publication_directory:
    publication_root = Path(publication_directory)
    descriptor_path = publication_root / "descriptor.json"
    descriptor_path.write_bytes(raw_wire)
    request_path = publication_root / "request.json"
    request_path.write_bytes(domain_codec.encode_json(raw_request))
    for point in ("before", "after"):
        store_root = publication_root / point
        store_root.mkdir()
        source = publication_root / f"{point}.gz"
        source.write_bytes(raw_compressed)
        attempt = subprocess.run(
            [
                sys.executable,
                "-I",
                str(root / "tests/support/subprocesses/interrupt_publish.py"),
                str(store_root),
                str(source),
                str(descriptor_path),
                str(request_path),
                point,
            ],
            cwd=publication_root,
            capture_output=True,
            text=True,
            check=False,
            timeout=30,
        )
        assert attempt.returncode == 75, attempt.stderr
        (staging,) = store_root.glob(".staging-*")
        RawFiles.verify(raw_descriptor, staging, request=raw_request.binding)
        store = RawStore(store_root)
        published = store_root / raw_descriptor.sha256
        if point == "before":
            try:
                store.open(descriptor=raw_descriptor, request=raw_request.binding)
            except FileNotFoundError:
                reader_state = "Missing: staging is invisible to readers"
            else:
                raise AssertionError("Reader accepted an unpublished object")
        else:
            with store.open(descriptor=raw_descriptor, request=raw_request.binding) as stream:
                assert stream.read() == raw_compressed
            assert published.stat().st_ino == staging.stat().st_ino
            source.unlink()
            reader_state = f"Complete and verified: {raw_descriptor.size_bytes} bytes"
        publication_rows.append((f"Exit {point} link", "1 verified orphan", reader_state))
        recovered = store.publish(source, descriptor=raw_descriptor, request=raw_request.binding)
        recovered_inode = recovered.stat().st_ino
        assert store.publish(source, descriptor=raw_descriptor, request=raw_request.binding) == recovered
        assert recovered.stat().st_ino == recovered_inode
        with store.open(descriptor=raw_descriptor, request=raw_request.binding) as stream:
            assert stream.read() == raw_compressed
        assert domain_codec.encode_json(raw_descriptor) == raw_wire
        staging.unlink()  # This child has exited; no active publisher owns the orphan.
        assert list(store_root.iterdir()) == [published]
        publication_rows.append(
            (f"Retry {point} link", "0 after orphan cleanup", "Verified; repeat reuses inode")
        )
    published.chmod(0o644)  # Inject corruption by an external writer in this temporary experiment.
    published.write_bytes(b"X" + raw_compressed[1:])
    for operation in ("open", "publish"):
        try:
            if operation == "open":
                store.open(descriptor=raw_descriptor, request=raw_request.binding)
            else:
                store.publish(source, descriptor=raw_descriptor, request=raw_request.binding)
        except ValueError as error:
            assert "SHA-256 mismatch" in str(error)
            publication_rows.append(
                (f"Corrupt existing object: {operation}", "0", "Rejected: SHA-256 mismatch")
            )
        else:
            raise AssertionError("Accepted a corrupt published object")
    assert published.read_bytes() == b"X" + raw_compressed[1:]
NotebookDisplay.show_table(
    ("Boundary", "Staging state", "Reader / retry outcome"), publication_rows
)
print(
    "NB 04: 2 abrupt exits; 2 reconciled retries; 2 conflict rejections; original descriptor and clocks retained."
)

Boundary,Staging state,Reader / retry outcome
Exit before link,1 verified orphan,Missing: staging is invisible to readers
Retry before link,0 after orphan cleanup,Verified; repeat reuses inode
Exit after link,1 verified orphan,Complete and verified: 145 bytes
Retry after link,0 after orphan cleanup,Verified; repeat reuses inode
Corrupt existing object: open,0,Rejected: SHA-256 mismatch
Corrupt existing object: publish,0,Rejected: SHA-256 mismatch


NB 04: 2 abrupt exits; 2 reconciled retries; 2 conflict rejections; original descriptor and clocks retained.


### Research interpretation

An interrupted save produces either an unavailable input or a complete verified file. Recovery reuses the original bytes; a damaged saved file is rejected. This protects the inputs your analysis consumes.

<details>
<summary>Detailed evidence and scope of this checkpoint</summary>

Before the link, only hidden staging exists and the reader reports a missing
object. After the link, the reader sees the complete verified compressed object,
while the interrupted call has not acknowledged durable completion. Retry either
publishes from the saved source or verifies the existing object and synchronizes
it again. Repetition preserves its inode; descriptors and original clocks remain
unchanged. Corrupt existing bytes fail both read and publish and are not replaced.

This boundary requires a caller-controlled, durably provisioned directory on a
local filesystem supporting hard links and file/directory synchronization. External
mutation/removal must be excluded during verified reads. Read-only file mode is
an accidental-write guard, not protection from the directory owner. Orphan staging
is ignored; remove it only with writers stopped. No acquisition, persistent
manifest, provider validity, dataset publication or empirical P1 result is claimed.
Section 13 separately exercises the later acquisition step; this section retains the MR 04 storage check.

</details>

## 13. Can I distinguish a failed download from usable data? (NB 05)

**Read the result:** A denied or damaged response must stay a failed attempt. An empty output directory here is not evidence that the market was quiet.

<details>
<summary>Implementation and checkpoint details</summary>

Use an injected HTTP response to fetch, validate gzip, publish and reopen the exact compressed bytes. Decode the saved trade with the retained Tardis and Binance decoders. The sample and clock values are synthetic. No network access occurs.

This demonstrates raw-byte acquisition; source permission, actual framing and historical timestamp units still need a permitted probe.

</details>

In [14]:
# standard library imports
import socket
from http.client import HTTPResponse, HTTPSConnection
from io import BufferedReader, BytesIO
from unittest.mock import patch

# project library imports
from applications.acquire import fetch_partition
from market_data.contracts import TransientFetchError


class NotebookResponse:
    """Parse a synthetic wire response with the standard HTTP parser, without connecting."""

    @staticmethod
    def parse(payload: bytes, status: int = 200) -> HTTPResponse:
        """Prepare one gzip HTTP response for the injected transport boundary."""
        wire = (
            f"HTTP/1.1 {status} Synthetic\r\nContent-Encoding: gzip\r\n"
            f"Content-Length: {len(payload)}\r\n\r\n"
        ).encode() + payload
        with socket.socket() as sock:
            response = HTTPResponse(sock)
        response.fp.close()
        response.fp = BufferedReader(BytesIO(wire))
        response.begin()
        return response


acquisition_manifest = replace(
    fixture_inspection_manifests["btc-fixture"],
    job=replace(
        fixture_inspection_manifests["btc-fixture"].job,
        mode="historical",
        pipeline="acquisition.fetch",
    ),
)
acquisition_request = SourceRequest.partition(acquisition_manifest)[0]
acquisition_payload = gzip.compress(
    (fixture_inspection_fixture_root / "btcusdt.txt").read_bytes(), mtime=0
)
acquisition_reports = []
with TemporaryDirectory(prefix="acquisition-fetch-notebook-") as directory:
    acquisition_store = RawStore(Path(directory))
    acquisition_times = iter(
        (datetime(2026, 9, 15, tzinfo=UTC), datetime(2026, 9, 15, 0, 0, 1, tzinfo=UTC))
    )
    with (
        patch.object(HTTPSConnection, "request", return_value=None),
        patch.object(
            HTTPSConnection, "getresponse", return_value=NotebookResponse.parse(acquisition_payload)
        ),
    ):
        acquired = fetch_partition(
            acquisition_request,
            store=acquisition_store,
            connection=HTTPSConnection("api.tardis.dev"),
            clock=acquisition_times.__next__,
            report=acquisition_reports.append,
            timeout_seconds=1.0,
            max_bytes=4096,
            permitted_rights_reference="synthetic-local-fixtures-v1",
        )
    with acquisition_store.open(descriptor=acquired, request=acquisition_request.binding) as saved:
        retained = saved.read()
    assert retained == acquisition_payload
    saved_line = gzip.decompress(retained).decode().removesuffix("\n")
    saved_decoder = TardisDecoder(
        BinancePayloadDecoder(acquisition_manifest.instruments),
        source_id=acquisition_request.instrument.source,
    )
    saved_trade = saved_decoder.decode_line(
        saved_line,
        source_sequence=0,
        local_received=acquired.ts_received,
        local_ingested=acquired.ts_ingested,
        source_digest=acquired.sha256,
    )
    assert saved_trade.price == Decimal("67540.01000000")
    assert saved_trade.original_capture_time == "2024-06-01T00:00:00.0041023Z"
    assert saved_trade.provenance.ts_received == acquired.ts_received
    NotebookDisplay.show_table(
        ("Acquisition fact", "Observed value"),
        [
            ("Run identity", acquisition_manifest.identity),
            ("Partition identity", acquired.request_identity),
            ("Retained gzip SHA-256", acquired.sha256),
            ("Compressed bytes", acquired.size_bytes),
            ("Source capture", saved_trade.original_capture_time),
            ("Exchange trade time", saved_trade.provenance.ts_event),
            ("Local receipt", acquired.ts_received),
            ("Durable ingestion", acquired.ts_ingested),
            ("Exact price / quantity", f"{saved_trade.price} / {saved_trade.quantity}"),
        ],
    )
    progress_rows = [json.loads(row) for row in acquisition_reports]
    NotebookDisplay.show_table(
        ("Stage", "Status", "Bytes"),
        [(row["stage"], row["status"], row["bytes"]) for row in progress_rows],
    )

Acquisition fact,Observed value
Run identity,1d182093367a997d3dad80c4ef1331301d473a9992ee31871aa1d7e981aea3fc
Partition identity,d81ff64b1ee69f90c2d4d149579fb3fb7db22fcf35258ab452afbf7395b2f876
Retained gzip SHA-256,e4a81e816955ccd1884b30986a6fea04c10b8c0eafd14233d0341cc6a0b19d26
Compressed bytes,145
Source capture,2024-06-01T00:00:00.0041023Z
Exchange trade time,2024-05-31 23:59:59.993000+00:00
Local receipt,2026-09-15 00:00:00+00:00
Durable ingestion,2026-09-15 00:00:01+00:00
Exact price / quantity,67540.01000000 / 0.00146000


Stage,Status,Bytes
fetch,started,0
store,received,145
store,stored,145


### Failure boundaries

Inject malformed/truncated gzip, an auth denial and a socket timeout. Each failure must emit a terminal status and leave storage empty.

In [15]:
acquisition_failures = []
for label, payload, status, expected in (
    ("Malformed gzip", b"not gzip", 200, ValueError),
    ("Truncated gzip", acquisition_payload[:-4], 200, ValueError),
    ("Authentication", b"synthetic denied", 401, PermissionError),
    ("Timeout", acquisition_payload, 200, TransientFetchError),
):
    with TemporaryDirectory(prefix="acquisition-fetch-rejection-") as directory:
        failure_reports = []
        error = TimeoutError("injected socket timeout") if label == "Timeout" else None
        with (
            patch.object(HTTPSConnection, "request", side_effect=error),
            patch.object(
                HTTPSConnection, "getresponse", return_value=NotebookResponse.parse(payload, status)
            ) as response_patch,
        ):
            try:
                fetch_partition(
                    acquisition_request,
                    store=RawStore(Path(directory)),
                    connection=HTTPSConnection("api.tardis.dev"),
                    clock=acquisition_times.__next__,
                    report=failure_reports.append,
                    timeout_seconds=1.0,
                    max_bytes=4096,
                    permitted_rights_reference="synthetic-local-fixtures-v1",
                )
            except expected as failure:
                assert list(Path(directory).iterdir()) == []
                terminal = json.loads(failure_reports[-1])
                assert terminal["status"] == "failed"
                acquisition_failures.append(
                    (label, type(failure).__name__, terminal["stage"], "No published bytes")
                )
            else:
                raise AssertionError(f"{label} unexpectedly succeeded")
            finally:
                response_patch.return_value.close()
NotebookDisplay.show_table(
    ("Injected failure", "Exception", "Failed stage", "Storage"), acquisition_failures
)

Injected failure,Exception,Failed stage,Storage
Malformed gzip,ValueError,fetch,No published bytes
Truncated gzip,ValueError,fetch,No published bytes
Authentication,PermissionError,fetch,No published bytes
Timeout,TransientFetchError,fetch,No published bytes


### Research interpretation

Successful acquisition preserves the original file and clocks. Malformed, denied or timed-out responses stay failed attempts. This synthetic exercise does not establish real provider access; use the historical notebook for the actual Binance sample.

<details>
<summary>Detailed evidence and scope of this checkpoint</summary>

The saved response preserves the gzip digest and exact price, quantity, source capture, exchange event and original local clocks. Successful progress ends only after durable publication. Malformed/truncated gzip, authentication and timeout failures publish nothing and emit failure status.

The transport validates HTTP/gzip integrity; semantic frame validation, disconnect iteration, coverage and canonical dataset publication remain later work. The socket timeout bounds inactivity rather than total wall time. At MR 05, no Tardis source probe was performed because its rights admission and retention were unresolved. The founder later switched P1 to Binance archives; [research-foundations.ipynb](research-foundations.ipynb) demonstrates actual acquisition. Historical eligibility and revision-availability claims remain unproven. See the [MR 05 record](https://github.com/lixuanguang/alpha-research-lab-private/blob/db6579258b6a86df6ac0b3a568aa8b78bddd54ae/docs/roadmap/mr-05-implementation.md). Section 14 adds MR 06 resume; this section remains the one-partition checkpoint.

</details>

## 14. Can I continue an interrupted collection? (NB 06)

**Read the result:** Resumed and uninterrupted collection should retain the same logical completed inputs. This does not remove duplicate economic trades within those inputs.

<details>
<summary>Implementation and checkpoint details</summary>

Question: can a fresh process finish interrupted acquisition without losing or repeating logical observations?
The installed pipeline persists a receipt before raw publication and appends a completion afterward.
The retained [synthetic process fixture](../tests/support/subprocesses/acquire.py) supplies three distinct gzip observations over fake HTTP. It exits abruptly at the second partition, before/after raw publication or checkpoint replacement; actual files, atomic operations and synchronization remain in use.

Compare the full resumed manifest with an uninterrupted run. Fixed UTC clocks make the documents byte-identical here. In an actual interrupted run, a pending receipt keeps its original receipt time and records ingestion when recovery reconfirms durability; completed clocks never change. No external source is contacted.

</details>

In [16]:
# project library imports
from applications.acquire import acquire
from market_data.acquisition import AcquisitionManifest
from market_data.adapters.providers.tardis import TardisTransport


class NotebookResume:
    """Drive retained synthetic process fixtures and inspect their saved checkpoints."""

    @staticmethod
    def process(destination: Path, run_path: Path, point: str) -> subprocess.CompletedProcess[str]:
        """Start the candidate installed interpreter outside the checkout."""
        return subprocess.run(
            [
                sys.executable,
                "-I",
                str(root / "tests/support/subprocesses/acquire.py"),
                str(destination),
                str(run_path),
                point,
            ],
            cwd=destination,
            capture_output=True,
            text=True,
            timeout=30,
            check=False,
        )

    @staticmethod
    def clock() -> datetime:
        """Provide the fixture's fixed UTC instant for deterministic comparisons."""
        return datetime(2026, 9, 15, tzinfo=UTC)


resume_run = replace(
    acquisition_manifest,
    job=replace(
        acquisition_manifest.job, end=acquisition_manifest.job.start + timedelta(minutes=3)
    ),
)
resume_rows = []
resume_failures = []
with TemporaryDirectory(prefix="acquisition-resume-notebook-") as directory:
    resume_root = Path(directory)
    run_path = resume_root / "run.json"
    run_path.write_bytes(domain_codec.encode_json(resume_run))
    baseline = resume_root / "baseline"
    (baseline / "raw").mkdir(parents=True)
    result = NotebookResume.process(baseline, run_path, "none")
    assert result.returncode == 0, result.stderr
    expected_manifest = (baseline / "manifest.json").read_bytes()
    expected_state = domain_codec.decode_json(expected_manifest, AcquisitionManifest)
    assert len(expected_state.completed) == 3 and expected_state.pending is None
    for point in ("before-raw", "after-raw", "before-checkpoint", "after-checkpoint"):
        destination = resume_root / point
        (destination / "raw").mkdir(parents=True)
        result = NotebookResume.process(destination, run_path, point)
        assert result.returncode == 75, result.stderr
        checkpoint = domain_codec.decode_json(
            (destination / "manifest.json").read_bytes(), AcquisitionManifest
        )
        raw_count = len([p for p in (destination / "raw").iterdir() if not p.name.startswith(".")])
        result = NotebookResume.process(destination, run_path, "none")
        assert result.returncode == 0, result.stderr
        assert (destination / "manifest.json").read_bytes() == expected_manifest
        fetches = (destination / "fetches.txt").read_text().splitlines()
        assert fetches == (["0", "1", "1", "2"] if point == "before-raw" else ["0", "1", "2"])
        result = NotebookResume.process(destination, run_path, "none")
        assert result.returncode == 0, result.stderr
        assert (destination / "fetches.txt").read_text().splitlines() == fetches
        resume_rows.append(
            (
                point,
                len(checkpoint.completed),
                checkpoint.pending is not None,
                raw_count,
                ", ".join(fetches),
                "3; exact manifest match; repeat skips all",
            )
        )
    NotebookDisplay.show_table(
        (
            "Exit boundary",
            "Completed before",
            "Pending receipt",
            "Raw objects before",
            "Fetched offsets",
            "After resume",
        ),
        resume_rows,
    )
    NotebookDisplay.show_table(
        ("Offset", "Request identity", "Gzip digest", "Original receipt"),
        [
            (
                request.offset_minutes,
                descriptor.request_identity[:16],
                descriptor.sha256[:16],
                descriptor.ts_received,
            )
            for request, descriptor in zip(
                SourceRequest.partition(resume_run), expected_state.completed
            )
        ],
    )
    for label in ("missing", "mutated"):
        destination = resume_root / label
        (destination / "raw").mkdir(parents=True)
        (destination / "manifest.json").write_bytes(expected_manifest)
        for original in (baseline / "raw").iterdir():
            (destination / "raw" / original.name).write_bytes(original.read_bytes())
        damaged = destination / "raw" / expected_state.completed[0].sha256
        if label == "missing":
            damaged.unlink()
        else:
            payload = damaged.read_bytes()
            damaged.write_bytes(b"X" + payload[1:])
        result = NotebookResume.process(destination, run_path, "none")
        assert result.returncode != 0 and not (destination / "fetches.txt").exists()
        expected_error = "FileNotFoundError" if label == "missing" else "SHA-256 mismatch"
        assert expected_error in result.stderr
        assert (destination / "manifest.json").read_bytes() == expected_manifest
        resume_failures.append(
            (f"Completed object {label}", expected_error, 0, "No source access; manifest retained")
        )
NotebookDisplay.show_table(("Failure", "Rejection", "Fetches", "Retained state"), resume_failures)
print(
    "NB 06: four process interruptions recovered; three unique completions; completed bytes fail closed."
)

Exit boundary,Completed before,Pending receipt,Raw objects before,Fetched offsets,After resume
before-raw,1,True,1,"0, 1, 1, 2",3; exact manifest match; repeat skips all
after-raw,1,True,2,"0, 1, 2",3; exact manifest match; repeat skips all
before-checkpoint,1,True,2,"0, 1, 2",3; exact manifest match; repeat skips all
after-checkpoint,2,False,2,"0, 1, 2",3; exact manifest match; repeat skips all


Offset,Request identity,Gzip digest,Original receipt
0,d81ff64b1ee69f90,7e0577039da97144,2026-09-15 00:00:00+00:00
1,8cfcae782190b37b,ca0da4d50e10da90,2026-09-15 00:00:00+00:00
2,b42bc791c5e8c887,594da3490f761c20,2026-09-15 00:00:00+00:00


Failure,Rejection,Fetches,Retained state
Completed object missing,FileNotFoundError,0,No source access; manifest retained
Completed object mutated,SHA-256 mismatch,0,No source access; manifest retained


NB 06: four process interruptions recovered; three unique completions; completed bytes fail closed.


### Terminal failures and bounded backoff

A transient transport exception alone permits retry. The tuple `(0.1, 0.2)` allows two additional attempts; the injected sleep records these delays without waiting. Authentication and schema failures stop on their first attempt. Each failure leaves an incomplete saved manifest. Storage and checkpoint failures also stop the invocation; reopen the manifest before retrying an uncertain write.


In [17]:
retry_rows = []
for label, failure, expected_attempts in (
    ("Authentication", PermissionError("synthetic denial"), 1),
    ("Response schema", ValueError("synthetic invalid gzip"), 1),
    ("Exhausted transient", TransientFetchError("synthetic unavailable"), 3),
):
    with TemporaryDirectory(prefix="acquisition-resume-retry-") as directory:
        destination = Path(directory)
        (destination / "raw").mkdir()
        waits, reports = [], []
        with patch.object(TardisTransport, "fetch", side_effect=failure) as fetch:
            try:
                acquire(
                    resume_run,
                    manifest_path=destination / "manifest.json",
                    store=RawStore(destination / "raw"),
                    connection=HTTPSConnection("api.tardis.dev"),
                    clock=NotebookResume.clock,
                    sleep=waits.append,
                    report=reports.append,
                    timeout_seconds=1.0,
                    max_bytes=4096,
                    permitted_rights_reference="synthetic-local-fixtures-v1",
                    retry_delays=(0.1, 0.2),
                )
            except type(failure):
                saved = AcquisitionManifest.read(
                    destination / "manifest.json",
                    run=resume_run,
                    store=RawStore(destination / "raw"),
                )
                assert not saved.completed and saved.pending is None
                assert fetch.call_count == expected_attempts
                assert waits == ([0.1, 0.2] if expected_attempts == 3 else [])
                retry_rows.append(
                    (
                        label,
                        fetch.call_count,
                        str(waits),
                        len(saved.completed),
                        "Incomplete; exception propagated",
                    )
                )
            else:
                raise AssertionError(f"{label} unexpectedly completed")
NotebookDisplay.show_table(
    ("Failure", "Attempts", "Backoff seconds", "Completed", "Run outcome"), retry_rows
)

Failure,Attempts,Backoff seconds,Completed,Run outcome
Authentication,1,[],0,Incomplete; exception propagated
Response schema,1,[],0,Incomplete; exception propagated
Exhausted transient,3,"[0.1, 0.2]",0,Incomplete; exception propagated


### Research interpretation

Restart verifies already completed inputs before continuing. Missing or changed files stop the run instead of being silently replaced. Recovering collection does not fill gaps in the provider's market history.

<details>
<summary>Detailed evidence and scope of this checkpoint</summary>

Resume verifies the entire completed prefix before new source access. A receipt without a published object is fetched again; published pending bytes are verified and reused. Checkpoints preserve one completion per selected request, and repeated complete runs fetch nothing. Missing or modified completed objects fail instead of being replaced. Authentication/schema failures are terminal; exhausted transient attempts retain incomplete state.

The manifest uses one caller-controlled file with atomic replacement and file/directory synchronization. One writer owns the file; it rewrites and validates the bounded completion prefix at each checkpoint. Tests inject process death and I/O failure, not physical power loss. Hidden staging files from dead writers are ignored. Original receipt clocks survive recovery; only unacknowledged ingestion is timestamped when durability is confirmed again.

This is synthetic development evidence. The Tardis access boundary recorded in MR 05 applies to these historical synthetic fixtures. [research-foundations.ipynb](research-foundations.ipynb) uses the later founder-selected Binance public archive; historical eligibility and revision availability remain unproven. Section 15 exercises the installed acquisition command and expanded telemetry added in MR 07. Frame iteration/semantic validation belong to later MRs; the synthetic observation bytes here are not a provider-data acceptance claim. See the [MR 06 record](https://github.com/lixuanguang/alpha-research-lab-private/blob/db6579258b6a86df6ac0b3a568aa8b78bddd54ae/docs/roadmap/mr-06-implementation.md).

</details>

## 15. What did the collection actually finish? (NB 07)

**Read the result:** Distinguish reported download activity from a saved completion. Progress describes collection work; it is not a data-quality score.

<details>
<summary>Implementation and checkpoint details</summary>

Question: can a saved command/configuration resume from another directory and expose truthful progress?
The child process executes the installed `alpha-acquire` entry point with copied configuration and the existing synthetic HTTP/process fixture. Real filesystem persistence remains active; no endpoint is contacted.

Inspect the last acknowledged checkpoint before an abrupt exit, a keyboard interruption and retry exhaustion. Resume each command, then repeat it: logical completions, bytes and original completed clocks must remain unchanged. Credentials are never passed on the command line.

</details>

In [18]:
# standard library imports
import shutil
from importlib.metadata import distribution, entry_points

# project library imports
from applications.acquire import main as acquisition_main

assert callable(acquisition_main)
assert (
    next(iter(entry_points(group="console_scripts", name="alpha-acquire"))).value
    == "applications.acquire.workflow:main"
)
command_distribution = distribution("alpha-research-lab")
command_script = Path(
    command_distribution.locate_file(
        next(item for item in command_distribution.files if item.name == "alpha-acquire")
    )
).resolve()
assert command_script.is_file()


class NotebookAcquisitionCommand:
    """Drive the installed console script with copied inputs and a synthetic HTTP boundary."""

    @staticmethod
    def process(
        destination: Path, arguments: list[str], point: str | None
    ) -> subprocess.CompletedProcess[str]:
        """Run a fresh outside-checkout process, optionally injecting HTTP and interruption.

        Args:
            destination: Existing directory containing copied inputs and the process fixture.
            arguments: Non-secret command arguments interpreted relative to that directory.
            point: Synthetic fault boundary, or None to execute the installed command directly.

        Returns:
            Captured process exit, stdout progress and stderr diagnostics.
        """
        if point is None:
            command = [str(command_script), *arguments]
        else:
            command = [
                sys.executable,
                "-I",
                str(destination / "acquire.py"),
                str(destination),
                str(destination / "run.json"),
                point,
                "--command",
                *arguments,
            ]
        return subprocess.run(
            command, cwd=destination, capture_output=True, text=True, timeout=30, check=False
        )


command_rows = []
command_failures = []
command_progress = []
with TemporaryDirectory(prefix="acquisition-command-notebook-") as directory:
    command_root = Path(directory)
    for point, expected_status in (("after-raw", 75), ("interrupt", 130), ("exhausted", 4)):
        destination = command_root / point
        (destination / "raw").mkdir(parents=True)
        configuration = Configuration.read(root / "configs/data-jobs/fixture-inspection.json")
        job = replace(configuration.jobs[0], mode="historical", pipeline="acquisition.fetch")
        job = replace(job, end=job.start + timedelta(minutes=3))
        configuration = replace(configuration, jobs=(job,))
        (destination / "jobs.json").write_bytes(domain_codec.encode_json(configuration))
        shutil.copyfile(root / "configs/reference/binance-spot.json", destination / "catalog.json")
        shutil.copyfile(root / "tests/support/subprocesses/acquire.py", destination / "acquire.py")
        command_run = configuration.resolve(job.name, catalog, implementation_identity=ImplementationIdentity.sha256())
        (destination / "run.json").write_bytes(domain_codec.encode_json(command_run))
        arguments = [
            "--config",
            "jobs.json",
            "--catalog",
            "catalog.json",
            "--job",
            job.name,
            "--manifest",
            "manifest.json",
            "--store",
            "raw",
            "--rights-reference",
            job.rights_reference,
            "--timeout-seconds",
            "1",
            "--max-bytes",
            "4096",
            "--retry-delays",
            "0",
            "0",
        ]
        interrupted = NotebookAcquisitionCommand.process(destination, arguments, point)
        assert interrupted.returncode == expected_status, interrupted.stderr
        records = [json.loads(line) for line in interrupted.stdout.splitlines()]
        terminal_before = json.loads(interrupted.stderr) if interrupted.stderr else records[-1]
        assert terminal_before["completed"] == 1
        prefix = domain_codec.decode_json(
            (destination / "manifest.json").read_bytes(), AcquisitionManifest
        )
        resumed = NotebookAcquisitionCommand.process(
            destination, arguments, "retry" if point == "interrupt" else "none"
        )
        assert resumed.returncode == 0 and not resumed.stderr
        resume_records = [json.loads(line) for line in resumed.stdout.splitlines()]
        completed = resume_records[-1]
        assert (
            completed["status"] == "complete" and completed["completed"] == completed["total"] == 3
        )
        assert completed["command_identity"] == records[0]["command_identity"]
        saved_manifest = (destination / "manifest.json").read_bytes()
        state = domain_codec.decode_json(saved_manifest, AcquisitionManifest)
        assert state.completed[: len(prefix.completed)] == prefix.completed
        fetch_log = (destination / "fetches.txt").read_bytes()
        repeated = NotebookAcquisitionCommand.process(destination, arguments, None)
        assert repeated.returncode == 0 and not repeated.stderr
        assert (destination / "manifest.json").read_bytes() == saved_manifest
        assert (destination / "fetches.txt").read_bytes() == fetch_log
        command_rows.append(
            (
                point,
                interrupted.returncode,
                terminal_before["completed"],
                terminal_before["retries"],
                completed["completed"],
                completed["persisted_bytes"],
                completed["last_persistence"],
                "unchanged; no fetch",
            )
        )
        if point == "interrupt":
            command_progress = resume_records
    for damage in ("mutation", "missing"):
        damaged = destination / "raw" / state.completed[0].sha256
        original_payload = damaged.read_bytes()
        if damage == "mutation":
            damaged.chmod(0o644)
            damaged.write_bytes(b"X" + original_payload[1:])
        else:
            damaged.unlink()
        rejected = NotebookAcquisitionCommand.process(destination, arguments, None)
        assert rejected.returncode == (3 if damage == "mutation" else 4)
        error = json.loads(rejected.stderr)
        assert error["stage"] == "resume" and error["completed"] is None
        assert (destination / "manifest.json").read_bytes() == saved_manifest
        command_failures.append(
            (
                damage,
                rejected.returncode,
                error["error_type"],
                error["completed"],
                "retained; rejected before fetch",
            )
        )
        damaged.write_bytes(original_payload)
    command_identity = completed["command_identity"]
    command_run_identity = completed["run_identity"]

NotebookDisplay.show_table(
    (
        "Interruption",
        "Exit",
        "Completed before",
        "Retries before",
        "After resume",
        "Durable bytes",
        "Last acknowledged ingestion",
        "Repeat",
    ),
    command_rows,
)
NotebookDisplay.show_table(
    ("Rejected input", "Exit", "Error", "Verified completions", "Manifest"), command_failures
)
NotebookDisplay.show_table(
    ("Identity", "Value"),
    [("Resolved job/config", command_run_identity), ("Saved command", command_identity)],
)

Interruption,Exit,Completed before,Retries before,After resume,Durable bytes,Last acknowledged ingestion,Repeat
after-raw,75,1,0,3,132,2026-09-16T08:55:06.787983Z,unchanged; no fetch
interrupt,130,1,0,3,132,2026-09-16T08:55:08.266699Z,unchanged; no fetch
exhausted,4,1,2,3,132,2026-09-16T08:55:09.771822Z,unchanged; no fetch


Rejected input,Exit,Error,Verified completions,Manifest
mutation,3,ValueError,None,retained; rejected before fetch
missing,4,FileNotFoundError,None,retained; rejected before fetch


Identity,Value
Resolved job/config,28f59b0374b4aab032363b25b76859f57392e8564930e7b7531aae56fd705f93
Saved command,9291f7fae1c55d1d80a91c4a0bbbbdc420aebc0ff52b2201f504d8aeb51a38b6


In [19]:
NotebookDisplay.show_table(
    ("Scope", "Stage", "Status", "Completed", "Retries", "Partition bytes", "Acknowledged bytes"),
    [
        (
            row["scope"],
            row["stage"],
            row["status"],
            row["completed"],
            row["retries"],
            row.get("bytes", "—"),
            row["persisted_bytes"],
        )
        for row in command_progress
    ],
)
assert any(row["status"] == "retry" for row in command_progress)
assert command_progress[-1]["retries"] == 1
print(
    "NB 07: installed entry point, flushed progress, interruption/resume, repeat and damaged-object rejection verified."
)

Scope,Stage,Status,Completed,Retries,Partition bytes,Acknowledged bytes
job,configuration,started,None,0,—,None
job,resume,verifying,None,0,—,None
job,resume,resumed,1,0,—,44
partition,fetch,started,1,0,0,44
partition,fetch,failed,1,0,0,44
partition,fetch,retry,1,1,—,44
partition,fetch,started,1,1,0,44
partition,store,received,1,1,44,44
partition,store,stored,1,1,44,44
partition,checkpoint,checkpointing,1,1,—,44


NB 07: installed entry point, flushed progress, interruption/resume, repeat and damaged-object rejection verified.


### Research interpretation

The completion count refers to files safely recorded for the requested collection. A repeated successful command preserves those inputs. Interrupted or failed commands retain inspectable progress; none of these statuses is a claim that the market history is complete.

<details>
<summary>Detailed evidence and scope of this checkpoint</summary>

The command prints flushed JSON progress before process exit. Job totals advance only after a durable completion checkpoint; partition `stored` alone does not add a completion. Resume reports its verified prefix, accepted compressed bytes and the original ingestion instant of the last acknowledged completion. Retry totals restart per invocation. A repeat of the same resolved input preserves the manifest and performs no source access.

Mutation and missing-byte failures report unknown verified totals while resume verification is incomplete. An abrupt process kill cannot emit a terminal record; its last flushed progress and saved checkpoint remain inspectable. Keyboard interruption returns 130; exhausted retries return 4 with the prior checkpoint. The saved command hash identifies resolved input, absolute destinations and non-secret limits, and stays stable across repeated invocations. Original completed clocks remain fixed; new/recovered ingestion uses the actual UTC clock.

This is synthetic installed-consumer evidence. [research-foundations.ipynb](research-foundations.ipynb) provides the real-download workflow. These synthetic command checks do not establish external source access; the historical notebook retains separate actual Binance acquisition evidence. No source coverage, semantic frame validation, accepted dataset or M1/P1 completion is claimed. See the [MR 07 record](https://github.com/lixuanguang/alpha-research-lab-private/blob/db6579258b6a86df6ac0b3a568aa8b78bddd54ae/docs/roadmap/mr-07-implementation.md). **NB 08** is now exercised in sections 5 and 7 of the historical notebook, including equal-time rows, empty CSV and absent archives.

[research-foundations.ipynb](research-foundations.ipynb) exercises actual Binance archive acquisition; the Tardis-specific access boundary is superseded for P1.

</details>

## 16. Can source order or missing input mislead me? (NB 08)

These tiny **synthetic Binance-format** archives preserve the saved-row acceptance
checks. They make no network requests and need no exchange account. This is a
reader reliability exercise, not the source used by the current research notebooks.

Read the output for equal-time rows in original order, empty content with unknown
coverage, explicit malformed-input rejection and missing-file failures. Empty or
missing input never proves that the market was quiet.


In [20]:
# project library imports
from configuration import Configuration
from market_data.adapters.providers.binance import BinanceArchiveRows
from market_data.contracts import SourceRequest
from reference_data.catalogs import CatalogSnapshot

real_configuration = Configuration.read(root / "configs/data-jobs/binance-archive.json")
real_catalog = DomainCodec().decode_json(
    (root / "configs/reference/binance-archive-spot.json").read_bytes(), CatalogSnapshot
)
real_run = real_configuration.resolve("btc-archive", real_catalog, implementation_identity=ImplementationIdentity.sha256())
(real_request,) = SourceRequest.partition(real_run)

In [21]:
# standard library imports
from zipfile import ZIP_DEFLATED, ZipFile

# project library imports
from market_data.contracts import RawObject


class _MR08SavedFixture:
    """Save tiny synthetic source objects through the same installed storage boundary."""

    @staticmethod
    def save(
        directory: Path, content: bytes, *, truncated: bool = False
    ) -> tuple[BinanceArchiveRows, RawStore]:
        """Bind exact synthetic ZIP bytes to the existing expected day and research bounds."""
        with BytesIO() as output:
            with ZipFile(output, "w", compression=ZIP_DEFLATED) as archive:
                archive.writestr(
                    f"{real_request.instrument.symbol}-trades-{real_request.start.date()}.csv",
                    content,
                )
            payload = output.getvalue()
        source = directory / "synthetic.zip"
        source.write_bytes(payload[:-12] if truncated else payload)
        origin = RawFiles.describe(
            source,
            request=real_request.binding,
            retrieval_reference="synthetic:nb08",
            ts_received=datetime(2026, 9, 16, tzinfo=UTC),
            ts_ingested=datetime(2026, 9, 16, tzinfo=UTC),
        )
        store = RawStore(directory)
        store.publish(source, descriptor=origin, request=real_request.binding)
        manifest = AcquisitionManifest(run=real_run, completed=(origin,))
        return BinanceArchiveRows.from_manifest(manifest)[0], store


archive_read_outcomes = []
with TemporaryDirectory(prefix="nb08-saved-archives-") as temporary:
    directory = Path(temporary)
    equal_time_bytes = (
        b"12,100.00,2.0,200.000,1717200000000,true,true\r\n"
        b"11,101.00,3.0,303.000,1717200000000,false,true\n"
    )
    reader, store = _MR08SavedFixture.save(directory, equal_time_bytes)
    equal_time_rows = tuple(reader.read(store, max_bytes=4096))
    assert [row.position for row in equal_time_rows] == [0, 1]
    assert [row.columns()[0] for row in equal_time_rows] == ["12", "11"]
    assert b"".join(row.payload for row in equal_time_rows) == equal_time_bytes
    NotebookDisplay.show_table(
        ("Position", "Trade ID", "Exchange time text", "Exact source bytes"),
        [
            (row.position, row.columns()[0], row.columns()[4], repr(row.payload))
            for row in equal_time_rows
        ],
    )
    NotebookDisplay.show_table(
        ("Raw origin / bounds", "Observed"),
        [
            ("ZIP SHA-256", equal_time_rows[0].origin.sha256),
            ("Request identity", equal_time_rows[0].origin.request_identity),
            ("Selection start", equal_time_rows[0].selection_start),
            ("Selection end (exclusive)", equal_time_rows[0].selection_end),
            ("Recorder coverage", reader.recorder_coverage),
        ],
    )
    empty, store = _MR08SavedFixture.save(directory, b"")
    assert tuple(empty.read(store, max_bytes=4096)) == ()
    archive_read_outcomes.append(
        ("Empty CSV", "0 rows; no completeness claim", empty.recorder_coverage)
    )
    for label, content, truncated in (
        ("Invalid UTF-8", b"\xff,1,2,3,4,true,true\n", False),
        ("Malformed CSV", b'1,"unclosed,2,3,4,true,true\n', False),
        ("Truncated ZIP", equal_time_bytes, True),
    ):
        invalid, store = _MR08SavedFixture.save(directory, content, truncated=truncated)
        try:
            tuple(invalid.read(store, max_bytes=4096))
        except ValueError as error:
            archive_read_outcomes.append((label, str(error), invalid.recorder_coverage))
        else:
            raise AssertionError(f"Accepted {label}")
    absent = BinanceArchiveRows.from_manifest(AcquisitionManifest(run=real_run))[0]
    assert empty.descriptor is not None
    (directory / empty.descriptor.sha256).unlink()
    for label, missing in (("Absent daily completion", absent), ("Missing stored ZIP", empty)):
        try:
            tuple(missing.read(store, max_bytes=4096))
        except FileNotFoundError:
            archive_read_outcomes.append((label, "FileNotFoundError", missing.recorder_coverage))
        else:
            raise AssertionError(f"Accepted {label}")
NotebookDisplay.show_table(("Case", "Observed outcome", "Recorder coverage"), archive_read_outcomes)

Position,Trade ID,Exchange time text,Exact source bytes
0,12,1717200000000,"b'12,100.00,2.0,200.000,1717200000000,true,true\r\n'"
1,11,1717200000000,"b'11,101.00,3.0,303.000,1717200000000,false,true\n'"


Raw origin / bounds,Observed
ZIP SHA-256,3ba3e578882d43a112f2f0e692c706a2a6cd9910d7ac6398a996e2277eaa648f
Request identity,c48117ff86b3e215ac330e0568c8019242beed75968c9fdd6ada366e82452cd9
Selection start,2024-06-01 00:00:00+00:00
Selection end (exclusive),2024-06-01 01:00:00+00:00
Recorder coverage,unknown


Case,Observed outcome,Recorder coverage
Empty CSV,0 rows; no completeness claim,unknown
Invalid UTF-8,invalid Binance CSV at row 0: 'utf-8' codec can't decode byte 0xff in position 0: invalid start byte,unknown
Malformed CSV,invalid Binance CSV at row 0: unexpected end of data,unknown
Truncated ZIP,Binance archive has invalid or truncated ZIP bytes,unknown
Absent daily completion,FileNotFoundError,unknown
Missing stored ZIP,FileNotFoundError,unknown


**Interpretation:** the preserved byte order and rejection examples establish the
reader boundary. They establish neither market completeness nor historical
availability to a strategy. The [MR 08 record](https://github.com/lixuanguang/alpha-research-lab-private/blob/db6579258b6a86df6ac0b3a568aa8b78bddd54ae/docs/roadmap/mr-08-implementation.md)
retains the earlier real Binance run. Continue with
[saved OKX and Bitfinex books](research-foundations.ipynb) for current source work.
